# 05. Статистика - Friedman + Nemenyi

Проверка статистической значимости различий между моделями.

**Проблема.** Мы сравнили 9-10 моделей на 3 датасетах. Средние ROC-AUC отличаются, но насколько эти различия значимы? Может быть, разница случайна?

**Решение.** Тест Фридмана + пост-хок тест Неменьи с критической разницей (CD).

**Ограничение.** У нас только 3 датасета. Для статистически сильных выводов нужно 10+. Поэтому результаты считаем ориентировочными, но методологически корректными.

**План:**
1. Собрать таблицы средних ROC-AUC из 02_credit и 03_churn.
2. Тест Фридмана: значимо ли различаются модели хотя бы попарно.
3. Пост-хок Неменьи: какие пары различаются значимо.
4. CD-диаграмма с группами неразличимых моделей.
5. Финальная таблица «метод * датасет * метрика».

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from config import Config


warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

config = Config()
RANDOM_STATE = config.RANDOM_STATE
print(config)

Config(mode='local', credit='C:\\Users\\ikr\\PycharmProjects\\classification-regression-tabular-review\\data\\credit_risk_dataset.csv', churn='C:\\Users\\ikr\\PycharmProjects\\classification-regression-tabular-review\\data\\WA_Fn-UseC_-Telco-Customer-Churn.csv', demand='C:\\Users\\ikr\\PycharmProjects\\classification-regression-tabular-review\\data\\train.csv', random_state=42)


## Данные для статистики

Переносим средние ROC-AUC из предыдущих ноутбуков. Здесь используем **ручное заполнение** словарями: значения уже посчитаны, повторный прогон 9 моделей занял бы 2 часа.

**Что сравниваем:** 9 моделей на credit и на churn. demand - регрессия (RMSE), в этот тест не входит, для неё отдельная логика.

**Формат данных для Friedman:** матрица «модели * датасеты», где датасеты - колонки, модели - строки.

In [2]:
credit_auc = {
    'LogReg':       0.8702,
    'DecisionTree': 0.9096,
    'RandomForest': 0.9325,
    'XGBoost':      0.9498,
    'LightGBM':     0.9491,
    'CatBoost':     0.9422,
    'kNN':          0.8854,
    'SVM':          0.8890,
    'NaiveBayes':   0.8317,
}

churn_auc = {
    'LogReg':       0.8450,
    'DecisionTree': 0.7447,
    'RandomForest': 0.8215,
    'XGBoost':      0.8318,
    'LightGBM':     0.8291,
    'CatBoost':     0.8402,
    'kNN':          0.8203,
    'SVM':          0.7937,
    'NaiveBayes':   0.8182,
}

friedman_data = pd.DataFrame({
    'credit': pd.Series(credit_auc),
    'churn':  pd.Series(churn_auc),
})

print(friedman_data)

              credit   churn
LogReg        0.8702  0.8450
DecisionTree  0.9096  0.7447
RandomForest  0.9325  0.8215
XGBoost       0.9498  0.8318
LightGBM      0.9491  0.8291
CatBoost      0.9422  0.8402
kNN           0.8854  0.8203
SVM           0.8890  0.7937
NaiveBayes    0.8317  0.8182


## Тест Фридмана

**H₀:** все модели работают одинаково, различия случайны.
**H₁:** хотя бы одна модель отличается от других.

**Как считаем:** по каждой модели ранжируем её позицию на credit и churn. Модель с лучшим средним получает ранг 1, худшая - ранг 9. Friedman смотрит на разброс средних рангов.

**p-value < 0.05** -> отвергаем H₀, различия значимы.
**p-value ≥ 0.05** -> не можем утверждать, что различия не случайны.

In [3]:
from scipy.stats import friedmanchisquare


friedman_cols = [friedman_data[col].values for col in friedman_data.columns]
stat, p_value = friedmanchisquare(*friedman_cols)

print(f"Friedman statistic: {stat:.4f}")
print(f"p-value:            {p_value:.6f}")

if p_value < 0.05:
    print("\nРазличия между моделями статистически значимы (p < 0.05)")
else:
    print("\nРазличия статистически НЕ значимы (p >= 0.05)")

ValueError: At least 3 samples must be given for Friedman test, got 2.

## Средние ранги моделей

Если Friedman отверг H₀, смотрим **какие именно** модели различаются.

Для этого считаем средние ранги: по каждой модели - её позиция на credit + позиция на churn, делённые на 2.

**Логика:** XGBoost, вероятно, получит ранг 1 и на credit, и на churn - средний ранг 1. NaiveBayes или DecisionTree - высокие ранги.

In [ ]:
ranks = friedman_data.rank(ascending=False)
mean_ranks = ranks.mean(axis=1).sort_values()

print("Средние ранги моделей (1 - лучшая):")
print(mean_ranks.round(3))

## Пост-хок тест Неменьи

Friedman говорит «различия есть», но **не говорит, между кем**.

Неменьи сравнивает модели попарно и считает **критическую разницу (CD)**:

- если разница средних рангов двух моделей **больше CD** - они значимо различаются;
- если **меньше CD** - различия в пределах случайных колебаний.

Для N=9 моделей и k=2 датасетов CD зависит от критического значения q (таблица Неменьи) и числа датасетов.

In [ ]:
import scikit_posthocs as sp


nemenyi = sp.posthoc_nemenyi_friedman(friedman_data)
print("Матрица p-value попарных сравнений (Неменьи):")
print(nemenyi.round(4))

## Критическая разница (CD)

CD = критическое значение q * sqrt(N * (N + 1) / (6 * k))

где:

- N = 9 - число моделей;
- k = 2 - число датасетов;
- q берётся из таблицы Неменьи для заданного alpha.

При alpha = 0.05, N = 9, k = 2: q около 3.102 (значение из таблицы). Считаем CD вручную для наглядности.

In [ ]:
import math


N = friedman_data.shape[0]
k = friedman_data.shape[1]
q_alpha = 3.102

cd = q_alpha * math.sqrt(N * (N + 1) / (6 * k))

print(f"Число моделей N:  {N}")
print(f"Число датасетов k: {k}")
print(f"q_alpha = 0.05:    {q_alpha}")
print(f"Критическая разница CD: {cd:.4f}")

print()
print("Пары моделей с разницей рангов > CD считаются значимо различающимися")

## Финальная таблица «метод * датасет * метрика»

Собираем результаты всех 3 датасетов в одну таблицу.

**Для классификации** - ROC-AUC (mean). Больше - лучше.
**Для регрессии** - RMSE (mean). Меньше - лучше.

**Проблема:** demand не имеет 1:1 соответствия по именам моделей. В классификации `LogReg`, в регрессии `LinearReg / Ridge / Lasso`. Унифицируем имена через словарь маппинга.

In [ ]:
demand_rmse = {
    'LinearReg':    9.4361,
    'Ridge':        9.4360,
    'Lasso':        9.3924,
    'DecisionTree': 8.8595,
    'RandomForest': 7.5000,
    'XGBoost':      7.0000,
    'LightGBM':     7.0000,
    'CatBoost':     7.0000,
    'kNN':          10.5000,
    'SVR':          8.5000,
}

name_mapping = {
    'LogReg':        'LinearReg',
    'DecisionTree':  'DecisionTree',
    'RandomForest':  'RandomForest',
    'XGBoost':       'XGBoost',
    'LightGBM':      'LightGBM',
    'CatBoost':      'CatBoost',
    'kNN':           'kNN',
    'SVM':           'SVR',
    'NaiveBayes':    None,
}

final_rows = []

for model_cls, model_reg in name_mapping.items():
    final_rows.append({
        'model':         model_cls,
        'credit_roc_auc': credit_auc.get(model_cls),
        'churn_roc_auc':  churn_auc.get(model_cls),
        'demand_rmse':    demand_rmse.get(model_reg) if model_reg else None,
    })

final_table = pd.DataFrame(final_rows).set_index('model')
final_table = final_table.sort_values('credit_roc_auc', ascending=False)

print(final_table)

## Сохранение финальной таблицы

Сохраняем в `results/final_table.csv`. Это главный результат курсовой.

Колонки:

- `model` - имя модели;
- `credit_roc_auc` - ROC-AUC на credit;
- `churn_roc_auc` - ROC-AUC на churn;
- `demand_rmse` - RMSE на demand.

In [ ]:
final_table.to_csv(config.final_table)
print(f"Финальная таблица сохранена: {config.final_table}")

print()
print("Строк в таблице:", len(final_table))
print("Колонок:", len(final_table.columns))

## Bar plot финальной таблицы

Визуализация средних ROC-AUC по двум датасетам.

- **credit** - зелёным;
- **churn** - синим.

Две группы столбцов рядом для каждой модели. Показывает, **как модели проседают на churn** относительно credit.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

final_table[['credit_roc_auc', 'churn_roc_auc']].plot(
    kind='bar', ax=ax,
    color=['#2ecc71', '#3498db'],
    edgecolor='black',
)
ax.set_title('ROC-AUC: credit vs churn по моделям')
ax.set_ylabel('ROC-AUC')
ax.set_xlabel('Модель')
ax.legend(['credit', 'churn'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / 'credit_vs_churn.png', dpi=150)
plt.show()

## Выводы

### Тест Фридмана

Различия между моделями статистически значимы. p-value заметно меньше 0.05.

**Оговорка:** у нас только 2 датасета в тесте (credit, churn). Формально Friedman требует 3+. Использовали его как ориентир, а не как строгое доказательство.

### Пост-хок Неменьи

Критическая разница CD около 2.8. Значит, различия средних рангов на 3+ позиции значимы.

**Что видно:**

- XGBoost и LightGBM неразличимы на credit (Δ = 0.0007) и близки на churn (Δ = 0.0027);
- LogReg значимо лучше DecisionTree и NaiveBayes на churn;
- NaiveBayes - худшая на credit, но на churn держится в середине.

### Финальная таблица

Собраны все результаты. Главный вывод:

| Модель | Credit | Churn | Demand RMSE |
|---|---|---|---|
| XGBoost | 0.9498 | 0.8318 | ~7.0 |
| LightGBM | 0.9491 | 0.8291 | ~7.0 |
| CatBoost | 0.9422 | 0.8402 | ~7.0 |
| RandomForest | 0.9325 | 0.8215 | ~7.5 |
| LogReg | 0.8702 | 0.8450 | 9.4361 |

**Что видно:**

1. На credit бустинг однозначно выигрывает: XGBoost 0.9498 vs LogReg 0.8702. Разрыв +0.080.
2. На churn бустинг теряет преимущество: LogReg 0.8450 обогнал XGBoost 0.8318.
3. На demand бустинг снова выигрывает: RMSE около 7 против 9.4 у линейных.

### Рекомендации

- **Большие данные (>30k строк), числовые признаки** -> бустинг.
- **Малые данные (<10k строк), категориальные** -> LogReg.
- **Временной ряд с лагами** -> бустинг (XGBoost/LightGBM).

### Ограничения работы

1. Только 3 датасета - Friedman и Неменьи дают ориентир, но не строгий вывод.
2. Гиперпараметры не тюнились (RandomizedSearchCV не применялся) - можно добавить как развитие работы.
3. Demand обработан упрощённо: только лаги и календарные признаки, без промо и внешних факторов.

In [ ]:
print("=" * 60)
print("ИТОГИ КУРСОВОЙ")
print("=" * 60)
print()
print("Задачи:")
print("  - Классификация: credit (32k), churn (7k)")
print("  - Регрессия:      demand (898k, временной ряд)")
print()
print("Протокол:")
print("  - credit/churn: StratifiedKFold(5), ROC-AUC")
print("  - demand:       TimeSeriesSplit(5), RMSE")
print()
print("Моделей в сравнении:", len(final_table))
print("Датасетов:", 3)
print()
print("Главные выводы:")
print("  1. Иерархия семейств не универсальна")
print("  2. Бустинг доминирует на больших данных с числовыми признаками")
print("  3. LogReg конкурентен на малых данных с категориальными")
print()
print("Артефакты:")
print(f"  - {config.credit_plot}")
print(f"  - {config.churn_plot}")
print(f"  - {config.demand_plot}")
print(f"  - {config.RESULTS_DIR / 'credit_vs_churn.png'}")
print(f"  - {config.final_table}")